In [0]:
-- first create a copy of same table 
CREATE TABLE Clean_Dmart AS
SELECT * FROM D_Mart;



select * from Clean_Dmart;



-- check schema of table 
describe table clean_dmart;



-- check duplicates 
with t as (
    select *, row_number() 
over(partition by name, Brand, Price, DiscountedPrice, Category, SubCategory, Quantity, Description,BreadCrumbs
order by name) as rn
from clean_dmart)
select * from t 
where rn > 1;



-- delete duplicates
/* due to databrick not allowing deleteaing from CTE clone_table with row_number() column as rn */
CREATE OR REPLACE TABLE clean_dmart AS
SELECT *,
       ROW_NUMBER() OVER (
           PARTITION BY name, Brand, Price, DiscountedPrice, Category, 
                        SubCategory, Quantity, Description, BreadCrumbs
           ORDER BY name
       ) AS rn
FROM d_mart;


-- now delete duplicates
DELETE FROM clean_dmart WHERE rn > 1;


-- validate changes 
select * from clean_dmart
where rn > 1;


-- check null 
SELECT 
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS name_nulls,
    SUM(CASE WHEN Brand IS NULL THEN 1 ELSE 0 END) AS brand_nulls,
    SUM(CASE WHEN Price IS NULL THEN 1 ELSE 0 END) AS price_nulls,
    SUM(CASE WHEN DiscountedPrice IS NULL THEN 1 ELSE 0 END) AS discountedprice_nulls,
    SUM(CASE WHEN Category IS NULL THEN 1 ELSE 0 END) AS category_nulls,
    SUM(CASE WHEN SubCategory IS NULL THEN 1 ELSE 0 END) AS subcategory_nulls,
    SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) AS quantity_nulls,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS description_nulls,
    SUM(CASE WHEN BreadCrumbs IS NULL THEN 1 ELSE 0 END) AS breadcrumbs_nulls
FROM clean_dmart;



select * from clean_dmart
where name is null or 
Brand is null or 
Price is null or
DiscountedPrice is null or
Category is null or
SubCategory is null or
Quantity is null or
Description is null or
BreadCrumbs is null;



--- first delete the records where only name is avaiable 
select count(*) from clean_dmart
where name is not null and
Brand is null and
Price is null and
DiscountedPrice is null and
Category is null and
SubCategory is null and
Quantity is null and
Description is null and
BreadCrumbs is null;


DELETE from clean_dmart
where name is not null and
Brand is null and
Price is null and
DiscountedPrice is null and
Category is null and
SubCategory is null and
Quantity is null and
Description is null and
BreadCrumbs is null;


-- check how many left 
select count(*) from clean_dmart
where name is not null and
Brand is null and
Price is null and
DiscountedPrice is null and
Category is null and
SubCategory is null and
Quantity is null and
Description is null and
BreadCrumbs is null;


-- check how many record are with missing price or discounted price or both
-- drop those record you missed both values 
select count(*) from clean_dmart
where price is null or DiscountedPrice is null;


select count(*) from clean_dmart
where price is null and DiscountedPrice is null;


delete from clean_dmart 
where price is null and DiscountedPrice is null;


-- check where brand is null 
select count(*) from clean_dmart
where Brand is null;


select count(*) from clean_dmart
where Brand is null and Category is not null;


-- fill brand with category where brand name is not available 
update clean_dmart
set Brand = Category
where Brand is null and Category Is not null;


-- check how many category & sub category is not available 
-- for category
select count(*) from clean_dmart
where Category is null;


-- for sub category
select count(*) from clean_dmart
where SubCategory is null;


-- for both 
select count(*) from clean_dmart
where Category is null and SubCategory is null;


-- for which value is missing 
select name,Category, SubCategory, BreadCrumbs from clean_dmart
where Category is null or SubCategory is null;


-- check if another order with same product is available
select name,Category, SubCategory, BreadCrumbs from clean_dmart
where name in (select name from clean_dmart
where Category is null or SubCategory is null);


-- what kind of category are there
select distinct category, subcategory from clean_dmart
order by Category asc;


-- update category sub category valus
update clean_dmart
set Category = "Home & Kitchen", subcategory ="Home & Kitchen"
where Category is null and SubCategory is null;


-- check null in Quantity
select count(*) from clean_dmart
where Quantity is null;


select * from clean_dmart
where Quantity is null;


-- check null in Quantity
select count(*) from clean_dmart
where Description is null;


select * from clean_dmart
where Description is null;


-- Update description
UPDATE clean_dmart
SET Description = 'No description available'
WHERE Description IS NULL;


-- check null in Quantity
select count(*) from clean_dmart
where breadcrumbs is null;


select * from clean_dmart
where breadcrumbs is null;


-- update Breadcrumbs
update clean_dmart
set BreadCrumbs = concat(Category,' > ',SubCategory)
where BreadCrumbs is null;


-- solve data typre 
select distinct Quantity
from clean_dmart;


describe  table clean_dmart;


-- check correctness of data 
select count(*) from clean_dmart
where Price < DiscountedPrice;

-- check consistency 
select distinct category from clean_dmart;

select distinct SubCategory from clean_dmart;



-- solve problem of Quantity issue
CREATE OR REPLACE VIEW dmart_quantity_clean AS
SELECT 
    Quantity,

    CASE 
        WHEN LOWER(Quantity) REGEXP 'kg|gm|g' THEN 'weight'
        WHEN LOWER(Quantity) REGEXP 'ml|l|litre' THEN 'volume'
        WHEN LOWER(Quantity) REGEXP 'pcs|unit|u|bags|tablets|wipes' THEN 'count'
        WHEN LOWER(Quantity) REGEXP 'x|pack|set' THEN 'pack'
        ELSE 'invalid'
    END AS quantity_type,


    CASE 
        WHEN LOWER(Quantity) LIKE '%kg%' THEN 'kg'
        WHEN LOWER(Quantity) LIKE '%gm%' OR LOWER(Quantity) LIKE '%g%' THEN 'g'
        WHEN LOWER(Quantity) LIKE '%ml%' THEN 'ml'
        WHEN LOWER(Quantity) LIKE '%l%' THEN 'l'
        WHEN LOWER(Quantity) REGEXP 'pcs|unit|u' THEN 'pcs'
        ELSE NULL
    END AS unit,

    CASE 
        WHEN LOWER(Quantity) REGEXP 'x\\s*[0-9]+' 
            THEN CAST(REGEXP_EXTRACT(Quantity, 'x\\s*([0-9]+)') AS INT)
        WHEN LOWER(Quantity) LIKE '%pack%' 
            THEN CAST(REGEXP_EXTRACT(Quantity, '[0-9]+') AS INT)
        ELSE 1
    END AS multiplier

FROM clean_dmart;


select * from dmart_quantity_clean
